In [1]:
import json
import pandas as pd

df = pd.read_csv("dataset-exemplos.csv", sep=";")

print(df.head())

     ID                                               Text   Label
0  D1-1  It is an approximation useful in chemistry, bu...   Human
1  D1-2  PET scanning, or Positron Emission Tomography,...    Meta
2  D1-3  Positron Emission Tomography (PET) scanning is...  Google
3  D1-4  Thermonuclear fusion is the process of combini...    Meta
4  D1-5  These nutrients are needed to keep bones, teet...   Human


In [2]:
df = df[["Text", "Label"]].copy()

# Remove rows with missing values, if any
df = df.dropna(subset=["Text", "Label"])

# Make sure text is treated as string
df["Text"] = df["Text"].astype(str)

In [3]:
from sklearn.preprocessing import LabelEncoder

# LabelEncoder converts class names into integer labels
encoder = LabelEncoder()
df["label"] = encoder.fit_transform(df["Label"])

# Print the class names to see the mapping
print("\nClasses:")
for i, class_name in enumerate(encoder.classes_):
    print(f"{i} -> {class_name}")


Classes:
0 -> Anthropic
1 -> Google
2 -> Human
3 -> Meta
4 -> OpenAI


In [5]:
# Separate input texts and target labels
texts = df["Text"]
labels = df["label"]

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create the vectorizer
# max_features=5000 keeps only the 5000 most important words
# stop_words="english" removes common English words such as "the", "is", etc.
vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

# Learn the vocabulary from the texts and transform them into TF-IDF vectors
# toarray() converts the sparse matrix into a dense NumPy array
X = vectorizer.fit_transform(texts).toarray()
y = labels.values


In [8]:
from sklearn.model_selection import train_test_split
# stratify=y helps keep class distribution similar in both sets
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [12]:

import torch
from torch.utils.data import Dataset
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, X, y):
        # Convert input features to float tensors
        self.X = torch.tensor(X, dtype=torch.float32)

        # Convert labels to integer tensors
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        # Return the total number of samples
        return len(self.X)

    def __getitem__(self, idx):
        # Return one feature vector and its label
        return self.X[idx], self.y[idx]


# Create train and validation datasets
train_dataset = TextDataset(X_train, y_train)
val_dataset = TextDataset(X_val, y_val)

# Create data loaders for mini-batch training
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


In [14]:
import torch.nn as nn

# Define a Feedforward Neural Network for text classification
class DNNClassifier(nn.Module):

    def __init__(self,input_size,num_classes):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_size,256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256,128),
            nn.ReLU(),

            nn.Linear(128,num_classes)

        )

    def forward(self,x):

        return self.network(x)

In [15]:
# Get the number of input features from TF-IDF
input_size = X.shape[1]

# Get the number of output classes
num_classes = len(encoder.classes_)

# Create the model
model = DNNClassifier(input_size, num_classes)

In [16]:
# Define the loss function for multi-class classification
criterion = nn.CrossEntropyLoss()

# Define the optimizer
# Adam is commonly used because it usually converges faster than standard SGD
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [17]:
epochs = 10

# Training loop
for epoch in range(epochs):

    # Set the model to training mode
    model.train()

    total_loss = 0

    # Iterate over the training batches
    for X_batch, y_batch in train_loader:

        # Reset gradients from the previous step
        optimizer.zero_grad()

        # Forward pass: compute predictions
        outputs = model(X_batch)

        # Compute the loss between predictions and true labels
        loss = criterion(outputs, y_batch)

        # Backward pass: compute gradients
        loss.backward()

        # Update model weights
        optimizer.step()

        # Accumulate loss for monitoring
        total_loss += loss.item()

    # Print the total loss at the end of each epoch
    print("Epoch", epoch, "Loss:", total_loss)

Epoch 0 Loss: 6.467143297195435
Epoch 1 Loss: 6.375478744506836
Epoch 2 Loss: 6.297908067703247
Epoch 3 Loss: 6.248585224151611
Epoch 4 Loss: 5.970075368881226
Epoch 5 Loss: 5.750744938850403
Epoch 6 Loss: 5.492914438247681
Epoch 7 Loss: 4.92037308216095
Epoch 8 Loss: 4.123624682426453
Epoch 9 Loss: 3.912290632724762


In [18]:
correct = 0
total = 0

# Set the model to evaluation mode
model.eval()

# Disable gradient computation because we are only evaluating
with torch.no_grad():

    # Iterate over the validation batches
    for X_batch, y_batch in val_loader:

        # Forward pass
        outputs = model(X_batch)

        # Get the predicted class index
        _, predicted = torch.max(outputs, 1)

        # Update total number of samples
        total += y_batch.size(0)

        # Count how many predictions are correct
        correct += (predicted == y_batch).sum().item()

accuracy = correct/total

print("Validation accuracy:",accuracy)

Validation accuracy: 0.4
